# UK Road Safety — Final Report

**Project:** Mapping and Modeling UK Road Safety — TEPP Phase 2 Portfolio Project
**Team (Group 8):** Leomary Rodriguez · Ye (Morris) Ou · Oluwafikunayomi Adeniji · Amina Jobarteh
**Data:** [UK Road Safety: Traffic Accidents and Vehicles](https://www.kaggle.com/datasets/tsiaras/uk-road-safety-accidents-and-vehicles) — UK Department for Transport, 2,047,256 accidents, 2005–2017

**The question this project answers:** *What makes a road accident more likely to be fatal?*

**Individual contributions for every team member are documented in the
[project README](../README.md).**


---

# Section 1 — EDA & Preprocessing Decisions

> Two files from the Department for Transport: about two million accidents, and the vehicles
> involved. Before we could learn anything we had to clean the data. The biggest problem was that
> **a lot of the missing information was not blank, it was typed out as words** like "Not known",
> which a computer reads as a real answer.
>
> The second thing we learned is a way of thinking. **Counting accidents tells you where people
> drive, not what is dangerous.** 80% of accidents happen in fine weather because that is when
> everyone drives. So we switched to measuring, for each group, *what share of its accidents ended
> in a death*. That change is what turned counts into findings.

## Preprocessing

| Problem | What we did |
|---|---|
| Missing values written as text — `'Not known'`, `'Data missing or out of range'`, `'Unknown'`, `'NA'` | Replaced with `np.nan`. Affects `Journey_Purpose_of_Driver` (~45%), `Junction_Control` (~37%), `Age_Band_of_Driver` (7.9%) and others. Uncleaned, "Not known" becomes the largest category in some columns. |
| `'Darkness - lighting unknown'` is a **real** answer, not a placeholder | Verified it survived the replacement — all 24,362 rows intact |
| `Carriageway_Hazards`, `Special_Conditions_at_Site` ~98% blank | **Kept.** Blank means *no hazard* — real information, not missing data |
| Impossible values: `Age_of_Vehicle` to 111 years, `Engine_Capacity` to 96,000cc | Flagged; capped before modelling |
| `Date`/`Time` stored as text | Parsed; `Month` and `Hour` derived. 156 unreadable times (0.008%) coerced to blank |
| Joining the files drops 118,797 vehicle rows | **All of them are from 2004**, the one year the accident file does not cover. Dropped deliberately rather than by accident |
| Code `109` in `Vehicle_Type`, 82,920 rows | It is **`Car`** under the pre-2005 coding scheme — it appears only in 2004, and 2004 has no rows labelled `Car` at all |
| Rows missing a clustering feature | Dropped (2.7%) rather than imputed — imputing invents conditions that were never observed |

## What we found

**The target is severely imbalanced** — Slight 84.73%, Serious 13.99%, **Fatal 1.29%**. A model that
always answers "Slight" scores **84.7% accuracy and finds zero fatal accidents**. This is why
accuracy is the wrong score.

The same problem gets *worse* if the question is narrowed to fatal against not-fatal, which is how
the supervised modelling frames it: always answering "not fatal" is correct for every Slight and
every Serious accident, so the do-nothing baseline rises to **98.71%**. Whichever way the target is
framed, accuracy rewards a model for ignoring the thing we care about.

**Counts measure exposure, not risk.** 80% of accidents were in fine weather, 64% at 30 mph. So every
comparison below is *within each group, what percentage were fatal*.

### The road environment

| Speed limit | % fatal | | Road type | % fatal | | Area | % fatal |
|---|---|---|---|---|---|---|---|
| 20 mph | 0.52% | | Roundabout | **0.31%** | | Urban | **0.71%** |
| 30 mph | 0.67% | | Single carriageway | 1.31% | | Rural | **2.35%** |
| 40 mph | 1.47% | | Dual carriageway | 1.75% | | | |
| 50 mph | 2.21% | | | | | | |
| **60 mph** | **3.15%** | | | | | | |
| 70 mph | **2.32%** | | | | | | |

**60 mph roads are more dangerous than 70 mph roads.** The reason is road design, not speed: 70 mph
means a motorway with the two directions separated by a barrier; 60 mph is the default for country
roads where cars can meet head-on. **This is why the model needs `Road_Type` and
`Urban_or_Rural_Area`, not just `Speed_limit`.**

### Lighting — the largest single effect

| Daylight | Darkness, lights lit | Darkness, lights off | **Darkness, no lighting** |
|---|---|---|---|
| 1.03% | 1.35% | 1.91% | **4.39%** |

**The comparison we trust most is 1.35% against 4.39%** — both are darkness, so lighting is close to
the only difference between them. That is cleaner than comparing dark against daylight.

### Weather — the opposite of what we expected

| Fog or mist | Fine + winds | Rain + winds | Fine | Rain | **Snow** |
|---|---|---|---|---|---|
| **2.15%** | 1.86% | 1.41% | 1.33% | 1.06% | **0.82%** |

**Crashes in snow are the least likely to be fatal.** Our explanation is that people drive slowly
when conditions are visibly bad, so the crash is more survivable. Fog fits — it is the deadliest, and
the hardest to judge from inside a warm car.

**This needs a warning wherever it is shown.** A chart that appears to say "snow is safer than
sunshine" will mislead anyone not given the explanation.

### Driver age — a U-shape, steeper at the old end

| 16–20 | **26–35** | 46–55 | 66–75 | **Over 75** |
|---|---|---|---|---|
| 1.19% | **1.17%** | 1.52% | 1.73% | **2.49%** |

Young drivers are the group people worry about, but it is the **oldest** drivers whose crashes are
most likely to be fatal. We tested two explanations and neither survived: restricting to cars only
makes the U *deeper* (0.94% → 2.40%), and restricting to urban cars keeps the shape (0.43% → 1.17%).
16–20 year olds even do a *higher* share of their driving on rural roads (45.4% vs 44.7%).

**The likely explanation is physical fragility, not worse driving** — but **we cannot confirm it**,
because the data records how bad the *accident* was, not who was hurt. That is the biggest caveat on
this finding.

### Briefly

- **Manoeuvres:** bends 3.18–3.55% and overtaking 2.63% are the worst — all country-road activities.
- **Vehicle age barely matters** — flat at 1.05–1.13% from new to fifteen years. It is the age of the
  *driver* that matters, not the car.
- **Vehicle make is not a safety rating.** DAF 4.47% vs Piaggio 0.59% looks dramatic, but DAF builds
  lorries and Piaggio builds scooters. **Comparing cars against cars the spread collapses from 7.6×
  to 1.9×.** `make` should stay out of the model.
- **Friday has the most accidents, Sunday the fewest** — an exposure result, driven by work journeys.
- **Accidents fell 34.6%** (198,735 in 2005 → 129,982 in 2017), **but the fatal share stopped
  improving in 2010** — 1.12% then, 1.29% by 2017. We cannot tell whether outcomes got worse or minor
  accidents stopped being reported. **So `Year` is not a neutral column** to hand a model.

## What this settled for modelling

**Include:** `Speed_limit`, `Light_Conditions`, `Urban_or_Rural_Area`, `Road_Type`,
`Age_Band_of_Driver`, `Vehicle_Manoeuvre`, plus `Number_of_Vehicles` and `Hour` on the clustering's
evidence.

**Exclude:** `make` (a proxy for vehicle type), `Age_of_Vehicle` (flat), `Journey_Purpose_of_Driver`
and `Junction_Control` (too incomplete), and `Number_of_Casualties` — an **outcome**, which would leak
the answer.

---

# Section 2 — Unsupervised Learning: Approach & Findings

**Notebook:** [`models/unsupervised_clustering.ipynb`](../models/unsupervised_clustering.ipynb) — Leomary Rodriguez & Oluwafikunayomi Adeniji

> Section 1 asked about one thing at a time; how dangerous is rain, how dangerous is darkness. But
> real accidents don't happen one condition at a time. They happen on a dark, wet, fast country road
> at eleven at night, all at once. So we asked a different question: **do UK accidents come in
> recognisable types?**
>
> To answer it we used **clustering**, which is a method for finding groups in data when nobody has
> told you what the groups are. We gave the computer ten pieces of information about the *situation*
> each accident happened in such as the kind of road, the speed limit, whether it was light or dark, the
> weather, the time of day, how many vehicles were involved and asked it to sort two million
> accidents into piles of similar situations. We did not tell it what to look for, and there was no
> right answer for it to find.
>
> **We never told it which accidents were fatal.** That rule is the whole reason the result means
> anything. If the computer had been shown the deaths, it would simply have sorted accidents by how
> bad they were, and we'd have learned nothing we didn't already know. So we kept severity out
> completely, formed the groups from circumstances alone, and only checked how deadly each group was
> **afterwards**.
>
> It found **five kinds of accident**, and we could name every one in ordinary English: fast rural
> roads, pile-ups, town roads after dark, the morning commute, and town roads in the afternoon.
>
> Then we looked at how each one had turned out. **The deadliest group is 4.6 times more likely to
> end in a death than the safest** — 2.65% against 0.57%. That gap matters because nothing about
> death went into building the groups. Sorting accidents purely by the circumstances they happened in
> also sorted them by how badly they ended, which means those circumstances genuinely carry
> information about survival.
>
> And the deadliest group it found — fast rural roads with nothing separating the two directions of
> traffic — is **exactly** the group our own analysis had flagged in Section 1, arrived at
> completely independently. Two different methods reaching the same answer is much stronger evidence
> than either one alone. It also found two things we had missed entirely, which are in the findings
> below.

## Method

**Ten features, all describing circumstances:** `Speed_limit`, `Number_of_Vehicles`, `Hour`,
`Road_Type`, `Light_Conditions`, `Weather_Conditions`, `Road_Surface_Conditions`,
`Urban_or_Rural_Area`, `Junction_Detail`, `Day_of_Week`.

**Left out on purpose:**

- `Accident_Severity` and `Number_of_Casualties` — **outcomes.** Including them would mean the
  clusters simply rediscover severity, and the feature we hand the supervised model would carry the
  answer inside it.
- `Junction_Control` and `Journey_Purpose_of_Driver` — 37% and 45% missing; we would be clustering on
  absence.
- `Latitude`/`Longitude` — would have produced a map of Britain rather than a description of driving
  conditions. "The North West" is not an accident profile.

**Setup:** KMeans, one-hot encoding for the nine categorical features, `StandardScaler` on the
numerics (unscaled, a 10 mph difference would outweigh the difference between daylight and an unlit
road). Fitted on a 150,000-row sample of the training set; train/test split `random_state=42`,
stratified on severity, **agreed with the supervised pair in advance** so both halves of the
modelling use the same split.

**What we expected, stated beforehand:** with only 1.29% of accidents fatal, every cluster would be
overwhelmingly Slight. We were not looking for a cluster of fatal crashes — we were looking for the
fatal *rate* to differ between clusters.

## The five accident profiles

| Cluster | Name | Share | Fatal % | vs average | What it is |
|---|---|---|---|---|---|
| **0** | **Fast rural roads** | 22.3% | **2.65%** | **2.0× worse** | ~62 mph single carriageways, away from junctions |
| **3** | **Multi-vehicle collisions** | 8.5% | **1.94%** | **1.5× worse** | 3.4 vehicles vs 1.7 elsewhere, ~42 mph |
| **4** | **Urban after dark** | 14.7% | 1.03% | 0.8× | 30 mph town T-junctions, ~7:30pm, lights lit |
| **1** | **The morning commute** | 19.9% | 0.98% | 0.8× | Urban 30 mph T-junctions, ~7am, daylight |
| **2** | **Urban afternoon** | 34.6% | **0.57%** | **0.4×** | The same roads as cluster 1, but ~3pm |

The average accident has about a 1.3% chance of being fatal. But that average hides a lot — crashes on fast rural roads are fatal 2.65% of the time, and crashes on urban roads in the afternoon only 0.57%. That's a 4.6-fold difference

## Findings

**1. The clustering confirmed the EDA independently.** Cluster 0 is exactly the fast rural
single-carriageway group Section 1 flagged — found by an algorithm given nothing but road, weather
and time. It also answers an open question from Section 1: the unlit roads, the rural roads and the
60 mph roads *are* substantially the same roads.

**2. Time of day matters on identical roads — new to us.** Clusters 1 and 2 share speed limit,
setting, junction type and light. Only the clock differs — 7am vs 3pm — and **the morning is nearly
twice as fatal.** We cannot explain it. Candidates are higher speeds on emptier roads, tiredness or
low sun glare, but those are guesses.

**3. Multi-vehicle collisions are their own profile — also new.** Cluster 3 is the only one defined by
a number rather than a place, and `Number_of_Vehicles` never came up in any of our four EDA
notebooks.

**4. Circumstances carry real information about outcome.** Rates vary 4.6× even though severity
played no part in forming the clusters. That is what justifies building a classifier at all.

---

**What happened when these labels reached the supervised model** — whether the cluster feature
improved it or not — is reported by Ye (Morris) Ou and Amina Jobarteh in Section 3, as part of their
modelling work.


---

# Section 3 — Supervised Models Tested & Compared

**Notebook:** [`models/supervised_regression.ipynb`](../models/supervised_regression.ipynb) — Ye (Morris) Ou & Amina Jobarteh

**What belongs here:** the target definition, the feature set, how the class imbalance was handled,
**every model tested and not just the winner**, the validation approach, and a comparison of the
model with and without the cluster label.

## Supervised Model (Regression)
- By Amina Jobarteh

### Core Setup & Framework

* **Target Definition:** The continuous target variable is the **accident fatality risk score** (a decimal value representing the probability of a crash resulting in a fatality).

* **Feature Set:** The raw feature matrix includes continuous and categorical data such as `Speed_limit`, `Hour`, `Number_of_Vehicles`, `Road_Type`, `Urban_or_Rural_Area`, and `Light_Conditions`.

* **Handling Imbalance:** Because fatal accidents are highly rare events, the target variable contains extreme imbalance. We handled this by keeping the target as a continuous risk scale and utilizing **tree-based regression models**, which naturally ignore major distribution shifts and focus tightly on localized data splits.

* **Validation Approach:** We utilized a strict **Train-Test Split**. Eighty percent of the data was used for training the models. The remaining 20% was locked away in a secret drawer as a hidden validation set ("final exam") to calculate true real-world performance.

---

### Models Tested 
We did not just evaluate our winner. We tested two advanced machine learning approaches side-by-side using the hidden test dataset:

* **Approach 1:** `HistGradientBoostingRegressor` (An agile model that builds trees sequentially to fix past mistakes).
* **Approach 2:** `RandomForestRegressor` (A model that builds 100 decision trees simultaneously and averages their outputs).


#### Performance Comparison

| Regression Model | Mean Absolute Error (MAE)  | Mean Squared Error (MSE)  | R² Score (Predictive Power)  |
| :--- | :--- | :--- | :--- |
| **1. HistGradientBoosting** | **0.02500** | **0.01252** | **0.01551** |
| **2. Random Forest** | 0.02508 | 0.01267 | 0.00308 |



---

### Key Insights & Interpretation

#### 1. Why are the R² scores low?
* **Real-World Chaos:** Traffic accidents are dictated by extreme random chaos. Predicting the exact row-by-row outcome of a single car crash is nearly impossible. Low R² scores are entirely standard for individual accident data.

#### 2. Row-by-Row Random Chaos
* **HistGradientBoosting** won the scoreboard because it adjusts its math aggressively for individual rows, capturing 5 times more row-by-row variance (R² = 0.01551).
* **Random Forest** protects itself from random chaos by leaning heavily on group averages, resulting in a lower individual score (R² = 0.00308).

#### 3. Model Results
* Despite different R² scores, **both models had high accuracy with the five accident environments** discovered by the unsupervised clustering team. When predictions are grouped back into their macro-clusters, both models track actual test fatality rates with stunning precision:

| Cluster | Environment Name | Actual Test Fatal % | HGB Predicted Risk % | RF Predicted Risk % |
| :--- | :--- | :--- | :--- | :--- |
| **Cluster 0** | Fast rural roads | **2.41%** | **2.38%** | **2.38%** |
| **Cluster 3** | Multi-vehicle collisions | **1.86%** | **1.91%** | **2.02%** |
| **Cluster 4** | Urban after dark | **1.07%** | **1.00%** | **1.01%** |
| **Cluster 1** | The morning commute | **0.98%** | **0.96%** | **0.98%** |
| **Cluster 2** | Urban afternoon | **0.54%** | **0.59%** | **0.58%** |

---

### Comparison: With vs. Without Cluster Labels
Our final breakthrough came from analyzing the model **with and without** the explicit cluster columns (`cluster_0.0`, `cluster_1.0`, etc.).

* **The Feature Importance Behavior:** When cluster labels were included, our feature importance scoreboard ranked raw metrics like `Speed_limit` (0.01261) and `Hour` (0.00593) at the very top. The pre-made cluster labels scored almost zero.

* **What this proves:** The tree-based models **do not need the explicit cluster labels to succeed**. Because the models are built out of complex tree splits, they easily look at the raw features (the ingredients) and independently reconstruct the exact macro-level environmental recipes found by the clustering team. 

* **Conclusion:** The unsupervised clustering pipeline was highly successful. It created a reliable real-world blueprint of accident environments that our final supervised model validated and predicted with immense accuracy.





## Supervised Model (Classification)

### Classification Approach

In addition to the regression analysis, I tested a classification approach to answer a different but related question: **can the accident characteristics help identify whether an accident was fatal?**

I converted `Accident_Severity` into a binary target:

* **1 — Fatal**
* **0 — Non-Fatal**, combining Slight and Serious accidents

Fatal accidents represent approximately **1.29%** of the dataset. Because the fatal class is rare, accuracy alone can give a misleading impression of model performance. For example, predicting every accident as non-fatal would still produce approximately **98.71% accuracy**, while identifying no fatal accidents.

For this reason, I evaluated the models using **accuracy, precision, recall, F1-score, and ROC-AUC**, with particular attention to the model's ability to identify fatal accidents.

### Models Tested

I tested two classification models using the same training and testing data:

1. **Logistic Regression** — used as a simple baseline for binary classification.
2. **HistGradientBoostingClassifier** — used to capture more complex relationships between the accident characteristics.

### Classification Results

| Model                          | Accuracy | Precision | Recall | F1-Score |    ROC-AUC |
| ------------------------------ | -------: | --------: | -----: | -------: | ---------: |
| Logistic Regression            |   98.71% |     0.00% |  0.00% |    0.00% |     0.7387 |
| HistGradientBoostingClassifier |   98.71% |    33.33% |  0.02% |    0.04% | **0.7717** |

### Results and Interpretation

Both models achieved **98.71% accuracy**, but the accuracy needs to be interpreted carefully because fatal accidents are very uncommon.

Logistic Regression did not identify any fatal accidents at the default classification threshold. Its precision, recall, and F1-score were therefore all **0.00%**. Its ROC-AUC of **0.7387**, however, indicates that the predicted probabilities still contained some ability to distinguish fatal from non-fatal accidents.

HistGradientBoostingClassifier achieved the same accuracy but a higher **ROC-AUC of 0.7717**. It also identified one fatal accident, giving it a precision of **33.33%**. However, its recall was only **0.02%**, so it still missed almost all fatal accidents.

The Tableau dashboard summarizes these results by comparing the two models across accuracy and ROC-AUC, and then across precision, recall, and F1-score. The charts make the main result clear: **the models have very high overall accuracy, but they are much less effective at identifying the rare fatal class.**

### Connection to the EDA and Clustering

The classification analysis connects to the earlier EDA and clustering findings.

The EDA showed that fatality rates vary across factors such as **speed limit, road type, rural or urban setting, lighting, weather, and driver age**. The clustering analysis then grouped accidents into five accident environments, with **Fast rural roads** having the highest observed fatal rate and **Urban afternoon** having the lowest.

The classification models used several of these accident characteristics to test whether they could help predict a fatal outcome for an individual accident.

This gives the project two complementary perspectives: **EDA and clustering help us understand which accident environments are associated with higher fatality rates, while classification tests how well those characteristics can be used for prediction.**

### Classification Conclusion

Based on ROC-AUC, **HistGradientBoostingClassifier was the stronger classification model**, scoring **0.7717** compared with **0.7387** for Logistic Regression.

However, neither model reliably identified fatal accidents at the default classification threshold. The extremely low recall shows that the current models are limited by the strong imbalance between fatal and non-fatal accidents.

The main conclusion is that the accident characteristics contain **some predictive information**, but the current classification approach is not sufficient for reliable individual fatality prediction. Future work could investigate different classification thresholds and methods for handling the class imbalance.


---

# Section 4 — Model Selection & Rationale

> The main decision was how many groups to sort accidents into. There is a standard score for this,
> and it said use **two**. We used **five** — and that was a judgement call, not something the maths
> decided, so we want to be open about it.
>
> Two groups splits the data into "town" and "countryside", which is a column we already had. The
> score was high because that split is clean, not because it was useful. So we judged each option on
> a different question: **does an extra group describe a genuinely different situation that is also
> genuinely more or less dangerous?**
>
> Everything below concerns the clustering. The supervised models are a separate piece of work with
> its own selection decisions, documented by the teammates who built them.


## Choosing the number of clusters

**The two standard checks disagreed.** The silhouette score was highest at **k=2 (0.229)** and fell to
**0.133 at k=5**. Inertia fell smoothly with no clear elbow.

But k=2 just splits urban from rural — something we already had as a column. **A high score for a
result we could get from a single `groupby` is not a useful result.** Silhouette measures how cleanly
separated clusters are, not how much they tell us, and on one-hot encoded data it almost always
favours fewer clusters.

So we tested every k against the question we actually cared about:

| k | Fatal spread | What the extra cluster added | Verdict |
|---|---|---|---|
| 2 | 3.7× | Urban vs rural — a column we already had | Fails |
| 3 | 3.8× | Split urban by time, but rates were 0.84% and 0.74% | Fails — same risk twice |
| 4 | 3.7× | Multi-vehicle collisions, 1.90% fatal | Passes |
| **5** | **4.6×** | Night-time urban; morning vs afternoon finally differ (0.98% vs 0.57%) | **Passes twice** |
| 6 | 5.0× | Split rural into 2.62% and 2.83% | Fails — same risk twice |

**Note that k=6 has the widest spread and we still rejected it** — the two rural halves are the same
risk labelled twice. Chasing the widest spread would have been the wrong rule.

**We chose k=5.** This is a judgement call and a different team could defend k=4 or k=6 from the same
plots.

**Why KMeans, and its weakness.** It is fast, its cluster centres are readable, and it assigns new
rows cleanly. But it assumes round clusters and continuous variables, and **nine of our ten features
are categorical** — which is also why the silhouette scores were low. `K-Prototypes`, which handles
mixed data natively, would be the honest next step.

---

> **Scope of this section.** The above covers model selection for the **unsupervised** work — the
> choice of k, and the choice of KMeans — by Leomary Rodriguez and Oluwafikunayomi Adeniji.
>
> **Model selection for the supervised models** — which models were compared, the metrics used, and
> why the final model was chosen over the alternatives — is documented by Ye (Morris) Ou and Amina
> Jobarteh alongside Section 3.


### Supervised Model Rationale (Regression)

I chose **HistGradientBoosting** and **Random Forest** because traditional linear models are too simple for chaotic traffic data. These tree-based models excel for four reasons:

* **Capture Complex Patterns:** Accident risk is not a straight line. It shifts constantly based on speed, time, and vehicle counts. These models use "Yes/No" splits to catch these messy real-world combinations.

* **Handle Missing Data:** Real datasets have empty cells. Traditional models crash on missing values, but `HistGradientBoosting` processes them automatically without dropping data.

* **Manage Rare Events:** Fatal crashes are thankfully rare. This creates a severe data imbalance. Tree models handle this by isolating specific risk groups rather than averaging the whole dataset.

* **Test Two Team Strategies:** We wanted to compare two different mathematical styles:
    * **HistGradientBoosting:** Builds trees one after another to fix past mistakes, creating highly precise individual risk guesses.
    * **Random Forest:** Builds 100 trees at once and averages their votes, creating stable predictions that resist random luck.

* **Metrics Used:*** I used Mean Squared Error and Mean Absolute Error to compare how much errors and mistakes the model can make with a test. I also used R-Squared (R^2) to see which model has a higher predictive power. 

---

# Section 5 — Detailed Results


> **The deadliest
> group of accidents is 4.6 times more likely to end in a death than the safest** — and the model
> sorted them without ever being told who died.

## The five clusters in full

| | Fast rural roads | Multi-vehicle | Urban after dark | Morning commute | Urban afternoon |
|---|---|---|---|---|---|
| **Cluster** | 0 | 3 | 4 | 1 | 2 |
| **Accidents** | 443,514 | 168,500 | 293,715 | 396,420 | 689,408 |
| **Fatal accidents** | **11,769** | 3,275 | 3,019 | 3,881 | 3,961 |
| **Share** | 22.3% | 8.5% | 14.7% | 19.9% | 34.6% |
| **Fatal %** | **2.65%** | **1.94%** | 1.03% | 0.98% | **0.57%** |
| **Mean speed limit** | **62 mph** | 42 mph | 31 mph | 31 mph | 31 mph |
| **Mean vehicles** | 1.70 | **3.39** | 1.72 | 1.68 | 1.68 |
| **Mean hour** | 13.2 | 13.3 | **19.6** | **7.1** | **15.0** |
| **Typical light** | Daylight | Daylight | **Darkness, lit** | Daylight | Daylight |
| **Setting** | **Rural** | Urban | Urban | Urban | Urban |
| **Typical junction** | Not at a junction | Not at a junction | T-junction | T-junction | T-junction |

Overall fatal rate 1.30%, across the 1,991,557 accidents that carry a cluster label.

**What is *not* distinctive here:** every cluster's most common road type is single carriageway,
because that is most of the road network. **The clusters are separated by the *combination* of speed,
setting, junction, light and time — not by any one column.** That is why cluster 0 could not have been
found by filtering on `Road_Type` alone.

**One number worth pulling out.** Fast rural roads are 22.3% of accidents but carry **11,769 of the
25,905 fatal accidents — 45% of every crash that killed someone.** No other profile comes close: the
next highest is urban afternoon at 3,961, from a group nearly twice as large. That is the case for
prioritising rural single carriageways.

*A fatal accident is one where at least one person died; the data does not record how many. So these
are counts of deadly crashes, not counts of deaths, and the true death toll is higher (§7).*

![Percent of accidents that were fatal, by cluster](figures/fatal_rate_by_cluster.png)

*Two profiles sit clearly above the 1.30% average line and three below it, the extremes 4.6× apart —
and the bars were built without the severity column ever being seen.*

## Cluster quality metrics

| k | Silhouette |
|---|---|
| 2 | **0.229** |
| 5 | **0.133** |

The score favours k=2. We chose k=5 anyway, and Section 4 sets out why — we report the metric that
disagrees with our choice rather than omitting it.

![Elbow and silhouette score by k](figures/elbow_silhouette.png)

*Left: inertia falls smoothly with no sharp elbow, so it does not settle the question either. Right:
the silhouette score peaks at k=2 and never recovers.*

A two-dimensional PCA projection of the clusters is in the clustering notebook. The groups overlap
heavily in it, which is expected for one-hot encoded data — every accident sits on the corner of a
high-dimensional cube rather than in a round cloud — and it is why we relied on the fatal-rate
evidence rather than the geometry.

![Clusters in two dimensions, PCA on 20,000 accidents](figures/clusters_pca.png)

*The axes have no real-world meaning. This is a visual check on whether the clusters occupy distinct
regions, not evidence in its own right.*

## Cross-check against the EDA

| EDA finding | Clustering result | Agree? |
|---|---|---|
| 60 mph roads deadliest at 3.15% | Cluster 0 averages 62 mph, 2.65% fatal | **Yes** |
| Rural 2.35% vs urban 0.71% | Cluster 0 rural and deadliest; the urban clusters safest | **Yes** |
| Unlit roads 4.39% | Cluster 0 is the rural group containing them | **Yes** |
| *Are these all the same roads?* — open question | They are | **Answered** |
| — | Morning 0.98% vs afternoon 0.57% on identical roads | **New** |
| — | Multi-vehicle collisions a distinct profile, 1.94% | **New** |

---

> **Results for the supervised models** — metrics, plots, and comparison across models — are reported
> by Ye (Morris) Ou and Amina Jobarteh alongside Section 3.

### Supervised Model (Regression)

#### Overall Performance Metrics
When grading the models row-by-row on the final exam, **HistGradientBoosting** outperformed Random Forest across every standard regression metric:

* **Mean Absolute Error (MAE):** HistGradientBoosting won (0.02500 vs. 0.02508). On average, its risk predictions were off by just 2.50%.

* **Mean Squared Error (MSE):** HistGradientBoosting won (0.01252 vs. 0.01267), meaning it made fewer massive guessing blunders.

* **R² Score (Predictive Power):** HistGradientBoosting won (0.01551 vs. 0.00308). Has a stronger predictive power than Random Forest.

#### Model Results 
When we group the final predictions back into the five unsupervised environments, both models show a high level accuracy:

| Cluster Label | Real-World Environment Recipe | Actual Test Fatal % | HGB Predicted Risk % | RF Predicted Risk % |
| :--- | :--- | :--- | :--- | :--- |
| **Cluster 0.0** | **Fast Rural Roads**  | **2.41%** | **2.38%** | **2.38%** |
| **Cluster 3.0** | **Multi-Vehicle** | **1.86%** | **1.91%** | **2.02%** |
| **Cluster 4.0** | **Urban After Dark**  | **1.07%** | **1.00%** | **1.01%** |
| **Cluster 1.0** | **The Morning Commute**  | **0.98%** | **0.96%** | **0.98%** |
| **Cluster 2.0** | **Urban Afternoon**  | **0.54%** | **0.59%** | **0.58%** |

#### Findings
The **HistGradientBoostingRegressor** is the stronger model due to it's higher predictive power and lower error metrics. 

However, the fact that *both* models independently matched the unsupervised team's risk profiles proves our feature engineering framework is stable and ready for real-world risk deployment.

---

# Section 6 — Dashboard Interpretation

**[View the live dashboard on Tableau Public →](https://public.tableau.com/app/profile/amina.jobarteh/viz/UKRoadSafety_17877843524320/Findings)**
Screenshots of each step are in [`dashboard/screenshots/`](../dashboard/screenshots).

> **In plain terms**
>
> The dashboard is built as a story in three steps, in the same order we did the work: what we found
> by exploring the data, what the clustering found, and what the models found. Someone with no
> technical background can click through it in a couple of minutes and reach the same conclusions we
> did, without reading a line of this report.

## How it is built

The workbook is a **Tableau Story** with three steps — *Exploratory Data Analysis*, *Unsupervised
Models & Techniques*, and *Supervised Models*. Every other worksheet is unpublished, so a visitor
lands on the story and sees only finished work.

The cluster labels reach Tableau through an **inner join** of `accidents_clean.csv` to
`accident_clusters.csv` on `Accident_Index`, giving 1,991,557 rows. The inner join matters: a left
join would keep the 55,699 accidents that have no cluster label, and they would pool into whichever
profile the calculated field defaulted to, distorting its rate.

## Step 1 — Exploratory findings

Four views, each a rate rather than a count, or a count and a rate side by side:

| View | Ties back to |
|---|---|
| **Most crashes happen in fine weather. Fog is most fatal.** — accident count beside fatal rate | §1.5 and §1.6. Fine weather holds 1,640,095 crashes at 1.33% fatal; fog holds 11,068 at 2.15% |
| **Light Conditions vs Accident Severity** | §1.6 — 4.4% fatal on unlit roads against 1.0% in daylight |
| **Serious and Fatal Accidents by Road Surface Condition** | §1.6 |
| **Accident Severity Over Time** | §1.6 — the long decline in total accidents |

**The weather view is the one that carries the method.** Putting the count bars next to the rate bars
shows the reader the illusion and the correction in the same image: fine weather dominates the
volume, and fog dominates the risk. Every later view assumes that lesson has landed.

## Step 2 — The five accident profiles

| View | Ties back to |
|---|---|
| **Accident Volume vs. Fatality Rate by Accident Profile** — count bars with the fatal rate overlaid, against a 1.30% reference line | §2.3 and §5. The largest profile, urban afternoon at 689,408 accidents, is also the safest at 0.57% |
| **Profile Fingerprints** — average speed limit, vehicle count and hour per profile, shaded by fatal rate | §5's cluster table. This is what makes the profile *names* defensible rather than asserted |
| **Morning vs Afternoon Fatal Rates** | §2.4, Finding 2 — 0.98% against 0.57% on identical roads |

This step is where the dashboard connects to the modelling rather than restating the EDA. The five
profiles on screen are the KMeans output from
[`unsupervised_clustering.ipynb`](../models/unsupervised_clustering.ipynb), not groups defined by
hand — and the fatal rates shown were calculated **after** clustering, exactly as §2.2 describes.

## Step 3 — Model results

The third step presents the supervised modelling: predicted against actual fatal rate per cluster,
an accuracy and ROC-AUC comparison across the classifiers, and a view of precision, recall and F1 on
the fatal class.

**These views and what they show are documented by Ye (Morris) Ou and Amina Jobarteh in Section 3**,
alongside the modelling that produced them.

## Decisions about what the dashboard does not show

**No vehicle brand view.** §1.6 found a 7.6× spread between manufacturers that collapses to 1.9× once
cars are compared with cars. The uncontrolled version would be a striking chart and a misleading one,
so it was left off deliberately (§8).

**Counts are never shown alone.** Every view carries a rate, because a count on its own measures how
much driving happens in a condition rather than how dangerous it is (§1.5).

**The weather view carries its caveat on the sheet.** Fog at 2.15% above snow at 0.82% reads as "snow
is safer than sunshine" to anyone who is not told why, so the explanation sits in the subtitle rather
than only in this report.

## Limitation carried onto the dashboard

Every rate shown is **the share of recorded crashes that were fatal** — not the chance of crashing.
The data contains no record of journeys where nothing went wrong (§7.1), so a profile with a low
fatal rate is not a safe profile to drive in; its crashes simply ended less badly. The dashboard
states this alongside the figures rather than leaving a viewer to infer it.

---

# Section 7 — Limitations


> **The biggest limitation: this data only contains crashes that happened.** There is no record of the
> millions of journeys where nothing went wrong. So when we say fast rural roads are deadliest, we
> mean *crashes there were more likely to end in a death* — not that you are more likely to crash
> there. We cannot measure the second thing at all.
>
> **The second: the data says how bad the accident was, not who got hurt.** So when we say over-75
> drivers have the deadliest crashes, we cannot tell whether the older driver died or someone in the
> other car did.

**No denominator.** Every rate here is *of the crashes that happened, what share were fatal* — never
*how likely a crash is*. This is why snow at 0.82% does not mean snow is safe, why cluster 2 being
safest does not make urban afternoons safe, and why vehicle make cannot be interpreted without
mileage data. Fixing it needs traffic volume data, which this dataset does not contain.

**Severity describes the accident, not a person.** This is the biggest caveat on the driver-age
finding, and it also affects the make finding and the odd 1.44% rate for parked vehicles.
Casualty-level STATS19 data would resolve all three.

**Missing data.** `Journey_Purpose_of_Driver` ~45%, `Junction_Control` ~37%, `Age_Band_of_Driver`
7.9%, `make` 5.1%. We dropped rather than imputed, and **those rows may not be missing at random** —
if police are less likely to record conditions at severe accidents, our rates would be biased in a
way we cannot detect from inside the data.

**Clustering choices.** KMeans is an imperfect fit for mostly-categorical data. **We chose k by
interpretability, not by metric**, and another team could defend k=4 or k=6. Cluster sizes are uneven
(34.6% vs 8.5%), and **Finding 2 has no explanation** — the morning/afternoon gap is real and
unexplained.

**Time and scope.** `Year` is not neutral — thirteen pooled years produce an average rate true in no
single year. The 2010 plateau has two incompatible explanations this data cannot separate. Scope is
Great Britain 2005–2017 and should not be transferred elsewhere or assumed current.

**Things that look like findings but are not.** Vehicle make is not a safety rating. Counts are not
risk. And **correlation is not cause anywhere here** — unlit roads are associated with fatal outcomes,
but we have not shown that installing lighting would reduce them, because unlit roads also differ in
speed and setting.

**Nothing in these sections is validated by a predictive model.** Every finding here comes from
describing the data — group rates and cluster profiles — not from testing whether those patterns hold
up as predictions on data the analysis had never seen. A pattern that is real in this dataset may or
may not generalise beyond it, and nothing in our work establishes that it does.

Supervised Model (Regression) Limitations

**Extreme Real-World Chaos:** Low R² scores prove car crashes are highly unpredictable. The model estimates general group risk but cannot predict individual luck.

**Imputed Data Bias:** Random Forest requires filling missing data. Using column medians adds artificial values, which introduces slight guessing bias.

**Severe Class Imbalance:** Fatal accidents are rare events. The model has few examples to study, making individual spikes hard to catch.

**Missing Environmental Context:** The dataset lacks real-time variables like bad weather, driver distraction, and holiday traffic surges.


---

# Section 8 — Recommendations

## For road safety authorities

| | Do this | Because |
|---|---|---|
| **1** | **Light the unlit roads** | 4.39% of crashes there are fatal, against 1.35% on lit roads. Both are darkness, so lighting is close to the only difference |
| **2** | **Separate the traffic on 60 mph single carriageways** | 3.15% fatal against 2.32% on motorways. This group is 22.3% of all accidents. Two independent methods found it |
| **3** | **Don't deprioritise urban roads** | Only 1.33% fatal, but 80% of all accidents happen there. Rural roads are deadliest; urban roads are where the volume is |
| **4** | **Convert junctions to roundabouts where feasible** | 0.31% fatal against 1.31% and 1.75% for single and dual carriageways — the largest margin of any road type |
| **5** | **Protect older drivers, don't restrict them** | Over-75s are 2.49% fatal against 1.17% for 26–35s. Most likely fragility, not driving — so this points to occupant protection and medical review at renewal |
| **6** | **Look into the morning commute** | 0.98% fatal at 7am against 0.57% at 3pm, on identical roads. We can't explain it |

**Two cautions.** Unlit roads are also mostly in rural areas and fast, so lighting is a marker of the deadliest
conditions rather than a guaranteed fix. And **we recommend against acting on the older-driver
finding without casualty-level data** — the effect is large and survives every control, but we cannot
confirm the fragility explanation from this dataset.


### Supervised modelling suggestion

For the classification analysis, examine the **Logistic Regression coefficients** to identify which accident characteristics have the strongest influence on the fatal versus non-fatal prediction. This would provide additional interpretation beyond the model performance metrics.